# BACKLOG para organizar singles/mixtape/EP/LP ao estilo Samply
Este caderno implementa um dashboard interativo (ipywidgets) para criar e gerir projetos musicais (single, EP, mixtape, LP), planear arte, gerir to-dos, letras, comparar mixes e emitir um sinal de conclusão baseado em diretrizes de alta qualidade.

# 1) Configurar Ambiente e Dependências
- Se alguma biblioteca faltar, rode `pip install` no seu ambiente (opcional).
- Este dashboard usa ipywidgets; no VS Code, a extensão Jupyter habilita os widgets automaticamente.

Requisitos principais (recomendado):
- ipywidgets, pandas, numpy
- pillow (PIL)
- librosa, soundfile, pyloudnorm (métricas de loudness)
- plotly e matplotlib (visualizações)
- pydub (opcional, para operações simples em áudio)

In [ ]:
# Imports centrais e setup (sem instalar pacotes aqui)
import sys, json, os, math, io, time, uuid, itertools, textwrap, copy
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import List, Dict, Optional, Any
from pathlib import Path

import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output, Audio, FileLink, Markdown

try:
    from PIL import Image, ImageOps, ImageDraw, ImageFont
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

# Dependências de áudio (podem faltar; tratamos como opcionais)
try:
    import librosa, soundfile as sf
    AUDIO_ANALYSIS_AVAILABLE = True
except Exception:
    AUDIO_ANALYSIS_AVAILABLE = False

try:
    import pyloudnorm as pyln
    LOUDNESS_AVAILABLE = True
except Exception:
    LOUDNESS_AVAILABLE = False

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

# Diretório base para persistência local do notebook
BASE_DIR = Path.cwd() / 'Projects'
BASE_DIR.mkdir(exist_ok=True)

def human_readable_seconds(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h:
        return f"{h:d}h {m:02d}m {s:02d}s"
    return f"{m:02d}m {s:02d}s"

In [ ]:
# 2) Modelo de Dados: Projeto, Faixa, Mix, Arte e Tarefas
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, List, Dict

class ProjectFormat(str, Enum):
    SINGLE = 'Single'
    EP = 'EP'
    MIXTAPE = 'Mixtape'
    LP = 'LP'
    CUSTOM = 'Custom'

class TaskStatus(str, Enum):
    TODO = 'To Do'
    DOING = 'Doing'
    DONE = 'Done'

class TaskPriority(str, Enum):
    LOW = 'Low'
    MEDIUM = 'Medium'
    HIGH = 'High'

@dataclass
class ArtworkSpec:
    concept: str = ''
    palette: str = ''
    refs: List[str] = field(default_factory=list)
    cover_path: Optional[str] = None

@dataclass
class MixVersion:
    name: str = 'v1'
    path: Optional[str] = None
    lufs: Optional[float] = None
    true_peak: Optional[float] = None
    rms: Optional[float] = None

@dataclass
class Track:
    title: str = 'Untitled'
    artist: str = ''
    audio_path: Optional[str] = None
    duration_sec: Optional[float] = None
    sample_rate: Optional[int] = None
    is_lead_single: bool = False
    disc: int = 1  # Para LP (multi-disc)
    mixes: List[MixVersion] = field(default_factory=list)
    lyrics: str = ''

@dataclass
class Task:
    title: str = ''
    category: str = 'mix'  # mix, master, art, release, marketing
    status: TaskStatus = TaskStatus.TODO
    priority: TaskPriority = TaskPriority.MEDIUM
    assignee: str = ''
    due: Optional[str] = None
    notes: str = ''

@dataclass
class Project:
    name: str = 'New Project'
    artist: str = ''
    fmt: ProjectFormat = ProjectFormat.SINGLE
    tracks: List[Track] = field(default_factory=list)
    tasks: List[Task] = field(default_factory=list)
    artwork: ArtworkSpec = field(default_factory=ArtworkSpec)
    created_at: float = field(default_factory=time.time)
    updated_at: float = field(default_factory=time.time)
    slug: Optional[str] = None

In [ ]:
# 3) Persistência e Estrutura de Pastas
import re, shutil, zipfile
from datetime import datetime

def slugify(value: str) -> str:
    value = re.sub(r'[^\w\-\s]', '', value).strip().lower()
    value = re.sub(r'[\s\-]+', '-', value)
    return value or 'project'

def project_root(proj: Project) -> Path:
    slug = proj.slug or slugify(proj.name)
    proj.slug = slug
    root = BASE_DIR / slug
    (root / 'tracks').mkdir(parents=True, exist_ok=True)
    (root / 'mixes').mkdir(parents=True, exist_ok=True)
    (root / 'art').mkdir(parents=True, exist_ok=True)
    (root / 'lyrics').mkdir(parents=True, exist_ok=True)
    (root / 'exports').mkdir(parents=True, exist_ok=True)
    return root

def save_project(proj: Project) -> Path:
    root = project_root(proj)
    proj.updated_at = time.time()
    data = asdict(proj)
    out_path = root / 'project.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return out_path

def load_project(slug: str) -> Optional[Project]:
    p = BASE_DIR / slug / 'project.json'
    if not p.exists():
        return None
    with open(p, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Reconstruir objetos
    proj = Project(**{k: v for k, v in data.items() if k not in ('tracks','tasks','artwork')})
    proj.artwork = ArtworkSpec(**data.get('artwork', {}))
    proj.tracks = [Track(**t) for t in data.get('tracks', [])]
    proj.tasks = [Task(**t) for t in data.get('tasks', [])]
    proj.slug = slug
    return proj

def backup_project_zip(proj: Project) -> Path:
    root = project_root(proj)
    ts = datetime.now().strftime('%Y%m%d-%H%M%S')
    zpath = root / f'backup-{ts}.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in root.rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(root))
    return zpath

def list_projects() -> List[str]:
    return sorted([p.name for p in BASE_DIR.iterdir() if p.is_dir() and (p / 'project.json').exists()])

In [ ]:
# 4) Dashboard com ipywidgets: criar/selecionar projeto e formato
# Estado global simples
STATE = {
    'project': None,  # type: Optional[Project]
    'current_track_index': 0,
}

fmt_dd = widgets.Dropdown(options=[(f.value, f) for f in ProjectFormat], description='Formato:')
name_tb = widgets.Text(description='Projeto:', placeholder='Nome do projeto')
artist_tb = widgets.Text(description='Artista:', placeholder='Nome do artista')
create_btn = widgets.Button(description='Criar', button_style='success')
existing_dd = widgets.Dropdown(options=['<novo>'] + list_projects(), description='Abrir:')
save_btn = widgets.Button(description='Salvar', icon='save', button_style='')
backup_btn = widgets.Button(description='Backup .zip', icon='cloud')

output_global = widgets.Output()

def handle_create(_):
    proj = Project(name=name_tb.value.strip() or 'New Project',
                   artist=artist_tb.value.strip(),
                   fmt=fmt_dd.value)
    # Padrões de faixas iniciais por formato
    n = 1 if proj.fmt == ProjectFormat.SINGLE else (4 if proj.fmt == ProjectFormat.EP else (8 if proj.fmt == ProjectFormat.MIXTAPE else 10))
    proj.tracks = [Track(title=f'Track {i+1}') for i in range(n)]
    STATE['project'] = proj
    save_project(proj)
    existing_dd.options = ['<novo>'] + list_projects()
    with output_global:
        clear_output()
        display(Markdown(f"Criado projeto: **{proj.name}** ({proj.fmt.value})"))

def handle_open(change):
    val = change['new']
    if val and val != '<novo>':
        proj = load_project(val)
        if proj:
            STATE['project'] = proj
            name_tb.value = proj.name
            artist_tb.value = proj.artist
            fmt_dd.value = proj.fmt
            with output_global:
                clear_output()
                display(Markdown(f"Aberto projeto: **{proj.name}**"))

def handle_save(_):
    if STATE['project'] is None:
        with output_global:
            clear_output()
            display(Markdown("Nenhum projeto carregado"))
        return
    proj = STATE['project']
    proj.name = name_tb.value.strip() or proj.name
    proj.artist = artist_tb.value.strip()
    proj.fmt = fmt_dd.value
    save_project(proj)
    with output_global:
        clear_output()
        display(Markdown(f"Salvo: **{proj.name}**"))

def handle_backup(_):
    if STATE['project'] is None:
        return
    z = backup_project_zip(STATE['project'])
    with output_global:
        clear_output()
        display(Markdown(f"Backup em {z}"), FileLink(str(z)))

create_btn.on_click(handle_create)
existing_dd.observe(handle_open, names='value')
save_btn.on_click(handle_save)
backup_btn.on_click(handle_backup)

top_bar = widgets.HBox([fmt_dd, name_tb, artist_tb, create_btn, save_btn, backup_btn, existing_dd])
display(top_bar, output_global)

In [ ]:
# 5) Gestão de Faixas: upload, metadados, ordem e agrupamento
tracks_box = widgets.VBox()
add_track_btn = widgets.Button(description='Adicionar Faixa', icon='plus')
up_btn = widgets.Button(description='Subir', icon='arrow-up')
down_btn = widgets.Button(description='Descer', icon='arrow-down')
del_btn = widgets.Button(description='Remover', icon='trash', button_style='danger')
track_selector = widgets.Dropdown(options=[], description='Faixa:')
lead_chk = widgets.Checkbox(value=False, description='Lead Single')
disc_int = widgets.BoundedIntText(value=1, min=1, max=10, description='Disco:')
audio_uploader = widgets.FileUpload(accept='.wav,.aiff,.aif,.mp3,.flac', multiple=False)
preview_output = widgets.Output()

def rebuild_track_selector():
    proj = STATE['project']
    if not proj:
        track_selector.options = []
        return
    track_selector.options = [ (f"{i+1:02d}. {t.title}", i) for i, t in enumerate(proj.tracks) ]
    if proj.tracks:
        track_selector.value = min(STATE['current_track_index'], len(proj.tracks)-1)
    else:
        STATE['current_track_index'] = 0

def current_track() -> Optional[Track]:
    proj = STATE['project']
    if not proj or not proj.tracks: return None
    i = STATE['current_track_index']
    if i < 0 or i >= len(proj.tracks): return None
    return proj.tracks[i]

def refresh_track_details():
    t = current_track()
    if not t: return
    lead_chk.value = t.is_lead_single
    disc_int.value = t.disc
    with preview_output:
        clear_output()
        if t.audio_path and Path(t.audio_path).exists():
            if AUDIO_ANALYSIS_AVAILABLE:
                try:
                    y, sr = librosa.load(t.audio_path, sr=None, mono=True)
                    dur = len(y)/sr
                    t.sample_rate = sr
                    t.duration_sec = dur
                    display(Markdown(f"**{t.title}** — {human_readable_seconds(dur)} @ {sr}Hz"))
                    display(Audio(data=y, rate=sr))
                except Exception as e:
                    display(Markdown(f"Falha ao carregar áudio: {e}"))
            else:
                display(Markdown(f"Áudio: {t.audio_path}"))

def handle_add_track(_):
    proj = STATE['project']
    if not proj: return
    proj.tracks.append(Track(title=f'Track {len(proj.tracks)+1}'))
    STATE['current_track_index'] = len(proj.tracks)-1
    rebuild_track_selector()
    refresh_track_details()
    save_project(proj)

def handle_order(delta: int):
    proj = STATE['project']
    if not proj or not proj.tracks: return
    i = STATE['current_track_index']
    j = i + delta
    if j < 0 or j >= len(proj.tracks): return
    proj.tracks[i], proj.tracks[j] = proj.tracks[j], proj.tracks[i]
    STATE['current_track_index'] = j
    rebuild_track_selector()
    save_project(proj)

def handle_delete(_):
    proj = STATE['project']
    if not proj or not proj.tracks: return
    i = STATE['current_track_index']
    proj.tracks.pop(i)
    STATE['current_track_index'] = max(0, i-1)
    rebuild_track_selector()
    refresh_track_details()
    save_project(proj)

def handle_track_change(change):
    STATE['current_track_index'] = change['new']
    refresh_track_details()

def handle_lead(change):
    t = current_track()
    if not t: return
    t.is_lead_single = change['new']
    save_project(STATE['project'])

def handle_disc(change):
    t = current_track()
    if not t: return
    t.disc = change['new']
    save_project(STATE['project'])

def handle_audio_upload(change):
    proj = STATE['project']
    t = current_track()
    if not proj or not t: return
    items = audio_uploader.value
    if not items: return
    (fname, meta), = items.items()
    # Salvar arquivo
    path = project_root(proj) / 'tracks' / fname
    with open(path, 'wb') as f:
        f.write(meta['content'])
    t.audio_path = str(path)
    save_project(proj)
    refresh_track_details()

add_track_btn.on_click(handle_add_track)
up_btn.on_click(lambda b: handle_order(-1))
down_btn.on_click(lambda b: handle_order(+1))
del_btn.on_click(handle_delete)
track_selector.observe(handle_track_change, names='value')
lead_chk.observe(handle_lead, names='value')
disc_int.observe(handle_disc, names='value')
audio_uploader.observe(handle_audio_upload, names='value')

rebuild_track_selector()
tracks_controls = widgets.HBox([add_track_btn, up_btn, down_btn, del_btn])
track_meta = widgets.HBox([track_selector, lead_chk, disc_int])
display(widgets.VBox([widgets.Label('Faixas'), tracks_controls, track_meta, audio_uploader, preview_output]))

In [ ]:
# 6) Planejamento de Capa (arte) e Checklist de Especificações
art_concept = widgets.Text(description='Conceito:', placeholder='Ideia principal da capa')
art_palette = widgets.Text(description='Paleta:', placeholder='Cores-chave (ex: #000, #FFF, dourado)')
art_refs = widgets.Textarea(description='Referências:', placeholder='URLs e observações')
art_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg', multiple=False)
art_output = widgets.Output()
spec_checks = widgets.VBox([
    widgets.Checkbox(description='3000x3000 px (≥300DPI)', value=False),
    widgets.Checkbox(description='Margens seguras / bleed', value=False),
    widgets.Checkbox(description='RGB sRGB IEC61966-2.1', value=False),
    widgets.Checkbox(description='Tipografia legível e legal', value=False),
])

def handle_art_upload(change):
    proj = STATE['project']
    if not proj: return
    items = art_upload.value
    if not items: return
    (fname, meta), = items.items()
    path = project_root(proj) / 'art' / fname
    with open(path, 'wb') as f:
        f.write(meta['content'])
    proj.artwork.cover_path = str(path)
    save_project(proj)
    with art_output:
        clear_output()
        display(Markdown(f"Capa guardada em: `{path}`"))
        if PIL_AVAILABLE:
            try:
                img = Image.open(path)
                display(img.resize((min(400, img.width), int(img.height*min(400, img.width)/img.width))))
            except Exception as e:
                display(Markdown(f"Falha ao abrir imagem: {e}"))

def persist_artwork(_):
    proj = STATE['project']
    if not proj: return
    proj.artwork.concept = art_concept.value
    proj.artwork.palette = art_palette.value
    proj.artwork.refs = [s.strip() for s in art_refs.value.splitlines() if s.strip()]
    save_project(proj)
    with art_output:
        clear_output()
        display(Markdown("Especificações de arte guardadas."))

art_upload.observe(handle_art_upload, names='value')
save_art_btn = widgets.Button(description='Salvar Arte', icon='save')
save_art_btn.on_click(persist_artwork)

display(widgets.VBox([widgets.Label('Capa / Arte'), art_concept, art_palette, art_refs, art_upload, spec_checks, save_art_btn, art_output]))

In [ ]:
# 7) To-do de Projeto e de Mix (prioridades, estados, notas)
task_title = widgets.Text(description='Tarefa:')
task_cat = widgets.Dropdown(options=['mix','master','art','release','marketing'], description='Categoria:')
task_pri = widgets.Dropdown(options=[p.value for p in TaskPriority], description='Prioridade:')
task_assignee = widgets.Text(description='Responsável:')
task_due = widgets.Text(description='Prazo:')
task_notes = widgets.Textarea(description='Notas:')
add_task_btn = widgets.Button(description='Adicionar', icon='plus')
tasks_output = widgets.Output()

def render_tasks():
    tasks_output.clear_output()
    with tasks_output:
        proj = STATE['project']
        if not proj or not proj.tasks:
            display(Markdown('Sem tarefas.'))
            return
        rows = []
        for i, t in enumerate(proj.tasks):
            cb = widgets.Checkbox(value=(t.status==TaskStatus.DONE), description=f"[{t.priority.value}] {t.title} ({t.category})")
            def on_change(change, idx=i):
                if change['name']=='value':
                    proj.tasks[idx].status = TaskStatus.DONE if change['new'] else TaskStatus.TODO
                    save_project(proj)
            cb.observe(on_change, names='value')
            rows.append(cb)
        display(widgets.VBox(rows))

def handle_add_task(_):
    proj = STATE['project']
    if not proj: return
    t = Task(title=task_title.value.strip() or 'Nova tarefa',
             category=task_cat.value,
             priority=TaskPriority(task_pri.value),
             assignee=task_assignee.value,
             due=task_due.value,
             notes=task_notes.value)
    proj.tasks.append(t)
    save_project(proj)
    task_title.value=''
    task_notes.value=''
    render_tasks()

add_task_btn.on_click(handle_add_task)
display(widgets.VBox([widgets.Label('To-do'), widgets.HBox([task_title, task_cat, task_pri]), widgets.HBox([task_assignee, task_due]), task_notes, add_task_btn, tasks_output]))
render_tasks()

In [ ]:
# 8) Letras: upload, edição, validação e versionamento
lyrics_selector = widgets.Dropdown(options=[], description='Faixa:')
lyrics_area = widgets.Textarea(description='Letras:', placeholder='Cole/edite as letras aqui...', layout=widgets.Layout(width='100%', height='200px'))
lyrics_upload = widgets.FileUpload(accept='.txt,.md', multiple=False)
save_lyrics_btn = widgets.Button(description='Salvar Letras', icon='save')
lyrics_output = widgets.Output()

def rebuild_lyrics_selector():
    proj = STATE['project']
    if proj:
        lyrics_selector.options = [(t.title, i) for i, t in enumerate(proj.tracks)]

def select_lyrics_track(change):
    proj = STATE['project']
    if not proj or not proj.tracks: return
    i = change['new']
    t = proj.tracks[i]
    lyrics_area.value = t.lyrics or ''

def handle_lyrics_upload(change):
    proj = STATE['project']
    if not proj: return
    items = lyrics_upload.value
    if not items: return
    (fname, meta), = items.items()
    text = meta['content'].decode('utf-8', errors='ignore')
    lyrics_area.value = text

def handle_save_lyrics(_):
    proj = STATE['project']
    if not proj: return
    idx = lyrics_selector.value
    if idx is None: return
    proj.tracks[idx].lyrics = lyrics_area.value
    # Persistir .md na pasta lyrics
    root = project_root(proj)
    p = root / 'lyrics' / f"{idx+1:02d}-{slugify(proj.tracks[idx].title)}.md"
    with open(p, 'w', encoding='utf-8') as f:
        f.write(lyrics_area.value)
    save_project(proj)
    with lyrics_output:
        clear_output()
        display(Markdown(f"Letras guardadas em {p}"))

lyrics_selector.observe(select_lyrics_track, names='value')
lyrics_upload.observe(handle_lyrics_upload, names='value')
save_lyrics_btn.on_click(handle_save_lyrics)

rebuild_lyrics_selector()
display(widgets.VBox([widgets.Label('Letras'), lyrics_selector, lyrics_area, lyrics_upload, save_lyrics_btn, lyrics_output]))

In [ ]:
# 9) Comparação de Mix: A/B, métricas e visualizações
mix_selector = widgets.Dropdown(options=[], description='Faixa:')
mix_upload = widgets.FileUpload(accept='.wav,.aiff,.aif,.mp3,.flac', multiple=True)
mix_ab_toggle = widgets.ToggleButtons(options=['A','B'], description='A/B:')
mix_output = widgets.Output()

def rebuild_mix_selector():
    proj = STATE['project']
    if proj:
        mix_selector.options = [(t.title, i) for i, t in enumerate(proj.tracks)]

def handle_mix_upload(change):
    proj = STATE['project']
    if not proj: return
    idx = mix_selector.value
    if idx is None: return
    t = proj.tracks[idx]
    for fname, meta in mix_upload.value.items():
        path = project_root(proj) / 'mixes' / fname
        with open(path, 'wb') as f:
            f.write(meta['content'])
        t.mixes.append(MixVersion(name=fname, path=str(path)))
    save_project(proj)
    show_mix()

def compute_loudness(y, sr):
    if not LOUDNESS_AVAILABLE:
        return None, None, None
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(y)
    peak = float(np.max(np.abs(y))) if y.size else None
    rms = float(np.sqrt(np.mean(y**2))) if y.size else None
    return loudness, peak, rms

def show_mix():
    proj = STATE['project']
    if not proj: return
    idx = mix_selector.value
    if idx is None: return
    t = proj.tracks[idx]
    if not t.mixes:
        with mix_output:
            clear_output()
            display(Markdown('Nenhuma versão de mix carregada.'))
        return
    choice = mix_ab_toggle.value
    mv = t.mixes[0] if choice=='A' else (t.mixes[1] if len(t.mixes)>1 else t.mixes[0])
    with mix_output:
        clear_output()
        display(Markdown(f"Mix selecionada: **{mv.name}**"))
        if AUDIO_ANALYSIS_AVAILABLE and mv.path and Path(mv.path).exists():
            try:
                y, sr = librosa.load(mv.path, sr=None, mono=True)
                dur = human_readable_seconds(len(y)/sr)
                lufs, peak, rms = compute_loudness(y, sr)
                display(Markdown(f"Duração: {dur} — SR: {sr}"))
                if LOUDNESS_AVAILABLE and lufs is not None:
                    display(Markdown(f"LUFS: {lufs:.2f} | Peak: {peak:.3f} | RMS: {rms:.3f}"))
                display(Audio(data=y, rate=sr))
                if PLOTLY_AVAILABLE:
                    # Forma de onda simples
                    xs = np.linspace(0, len(y)/sr, num=min(5000, len(y)))
                    ys = y[::max(1, len(y)//len(xs))][:len(xs)]
                    fig = go.Figure()
                    fig.add_scatter(x=xs, y=ys, mode='lines', name='Waveform')
                    fig.update_layout(height=200, title='Forma de onda')
                    fig.show()
            except Exception as e:
                display(Markdown(f"Falha ao analisar mix: {e}"))

mix_selector.observe(lambda c: show_mix(), names='value')
mix_upload.observe(handle_mix_upload, names='value')
mix_ab_toggle.observe(lambda c: show_mix(), names='value')
rebuild_mix_selector()
display(widgets.VBox([widgets.Label('Comparação de Mix'), widgets.HBox([mix_selector, mix_ab_toggle]), mix_upload, mix_output]))

In [ ]:
# 10) Diretrizes e Score de Qualidade com Sinal de Conclusão
quality_checks = widgets.VBox([
    widgets.Checkbox(description='Clipping: 0 ocorrências', value=False),
    widgets.Checkbox(description='Headroom adequado (true peak ≤ -1dBTP)', value=False),
    widgets.Checkbox(description='LUFS alvo atingido (ex.: -14 LUFS single, -8 a -10 club)', value=False),
    widgets.Checkbox(description='Compatibilidade mono OK', value=False),
    widgets.Checkbox(description='Balance tonal consistente', value=False),
])
score_bar = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description='Score:')
signal_html = widgets.HTML(value='<b style="color:red">Pendente</b>')

def compute_score() -> float:
    proj = STATE['project']
    if not proj: return 0.0
    # Peso simples: 40% tarefas concluídas, 40% diretrizes, 20% ativos completos (capa + mix final + letras)
    # Tarefas
    if proj.tasks:
        done = sum(1 for t in proj.tasks if t.status==TaskStatus.DONE)
        task_ratio = done/len(proj.tasks)
    else:
        task_ratio = 0.0
    # Diretrizes
    checks = quality_checks.children
    qual_ratio = sum(1 for c in checks if getattr(c, 'value', False))/len(checks) if checks else 0.0
    # Ativos
    cover_ok = bool(proj.artwork.cover_path)
    tracks_ok = all(t.audio_path for t in proj.tracks) if proj.tracks else False
    lyrics_ok = all(bool(t.lyrics.strip()) for t in proj.tracks) if proj.tracks else False
    assets_ratio = (int(cover_ok) + int(tracks_ok) + int(lyrics_ok))/3.0
    return 40*task_ratio + 40*qual_ratio + 20*assets_ratio

def refresh_quality(_=None):
    s = compute_score()
    score_bar.value = s
    if s >= 85.0:
        signal_html.value = '<b style="color:green">Concluído (Alta Qualidade)</b>'
    elif s >= 60.0:
        signal_html.value = '<b style="color:orange">Quase lá</b>'
    else:
        signal_html.value = '<b style="color:red">Pendente</b>'

for cb in quality_checks.children:
    cb.observe(refresh_quality, names='value')

display(widgets.VBox([widgets.Label('Qualidade'), quality_checks, score_bar, signal_html]))
refresh_quality()

In [ ]:
# 11) Exportação: metadados, assets e relatórios
export_btn = widgets.Button(description='Exportar Metadados', icon='download')
report_btn = widgets.Button(description='Gerar Relatório (Markdown)', icon='file')
export_output = widgets.Output()

def handle_export(_):
    proj = STATE['project']
    if not proj: return
    root = project_root(proj)
    # Metadados CSV simples
    csv_path = root / 'exports' / 'metadata.csv'
    rows = []
    for i, t in enumerate(proj.tracks):
        rows.append({'index': i+1, 'title': t.title, 'artist': proj.artist, 'lead_single': t.is_lead_single, 'disc': t.disc, 'duration_sec': t.duration_sec or ''})
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    with export_output:
        clear_output()
        display(Markdown(f"Metadados exportados em {csv_path}"), FileLink(str(csv_path)))

def handle_report(_):
    proj = STATE['project']
    if not proj: return
    root = project_root(proj)
    md_path = root / 'exports' / 'report.md'
    text = [f"# Relatório: {proj.name}", f"Artista: {proj.artist}", '', '## Faixas']
    for i, t in enumerate(proj.tracks):
        text.append(f"- {i+1:02d}. {t.title} (lead={t.is_lead_single}, disc={t.disc})")
    text.append('')
    text.append('## Tarefas')
    for t in proj.tasks:
        text.append(f"- [{ 'x' if t.status==TaskStatus.DONE else ' '}] ({t.priority.value}) {t.title} — {t.category}")
    text.append('')
    text.append('## Arte')
    text.append(f"- Conceito: {proj.artwork.concept}")
    text.append(f"- Paleta: {proj.artwork.palette}")
    if proj.artwork.cover_path:
        text.append(f"- Capa: {proj.artwork.cover_path}")
    text.append('')
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(text))
    with export_output:
        clear_output()
        display(Markdown(f"Relatório em {md_path}"), FileLink(str(md_path)))

export_btn.on_click(handle_export)
report_btn.on_click(handle_report)
display(widgets.HBox([export_btn, report_btn]), export_output)

In [ ]:
# 12) Autosave, Versionamento Simples e Backup
autosave_chk = widgets.Checkbox(value=True, description='Autosave')
autosave_output = widgets.Output()

def autosave():
    if autosave_chk.value and STATE['project'] is not None:
        save_project(STATE['project'])
        with autosave_output:
            clear_output()
            display(Markdown('Autosave OK'))

display(widgets.HBox([autosave_chk]), autosave_output)

In [ ]:
# 13) Testes Rápidos e Validações
def smoke_test():
    # Criar projeto de teste (não salvar em disco)
    p = Project(name='Teste', artist='Art', fmt=ProjectFormat.EP)
    p.tracks = [Track(title='Faixa 1'), Track(title='Faixa 2')]
    p.tasks = [Task(title='Mix kick', status=TaskStatus.DONE), Task(title='Arte WIP', status=TaskStatus.DOING)]
    s = 0.0
    STATE['project'] = p
    return compute_score()
display(Markdown('Smoke test score:'), smoke_test())

In [ ]:
# UI: Zoom 80% e barras fixas (top/bottom)
from IPython.display import HTML
HTML("""
<style>
/***** Zoom global *****/
body { zoom: 0.8; }

/***** Barra superior e inferior fixas (dentro da área do notebook) *****/
#fixed-top-bar, #fixed-bottom-bar {
  position: sticky;
  z-index: 999;
  background: white;
  border: 1px solid #e5e7eb;
  padding: 6px 10px;
}
#fixed-top-bar { top: 0; }
#fixed-bottom-bar { bottom: 0; }
</style>
""")

In [ ]:
# Tabs/Accordion: agrupar seções em navegação
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

def section_tracks_ui_compact():
    box = widgets.VBox([widgets.Label('Faixas (compacto) — use a seção 5 para recursos completos.'), track_selector, lead_chk, disc_int, audio_uploader, preview_output])
    return box

def section_art_ui_compact():
    box = widgets.VBox([widgets.Label('Arte (compacto) — use a seção 6 para recursos completos.'), art_concept, art_palette, art_refs, art_upload, spec_checks, art_output])
    return box

def section_tasks_ui_compact():
    box = widgets.VBox([widgets.Label('To-do (compacto) — use a seção 7 para recursos completos.'), tasks_output])
    return box

def section_lyrics_ui_compact():
    box = widgets.VBox([widgets.Label('Letras (compacto) — use a seção 8 para recursos completos.'), lyrics_selector, lyrics_area, lyrics_output])
    return box

def section_mix_ui_compact():
    box = widgets.VBox([widgets.Label('Mix A/B (compacto) — use a seção 9 para recursos completos.'), widgets.HBox([mix_selector, mix_ab_toggle]), mix_upload, mix_output])
    return box

def section_quality_ui_compact():
    box = widgets.VBox([widgets.Label('Qualidade (compacto) — use a seção 10 para controles.'), quality_checks, score_bar, signal_html])
    return box

def section_export_ui_compact():
    box = widgets.VBox([widgets.Label('Exportação (compacto) — use a seção 11 para mais.'), widgets.HBox([export_btn, report_btn]), export_output])
    return box

tabs = widgets.Tab(children=[
    section_tracks_ui_compact(),
    section_art_ui_compact(),
    section_tasks_ui_compact(),
    section_lyrics_ui_compact(),
    section_mix_ui_compact(),
    section_quality_ui_compact(),
    section_export_ui_compact(),
])
titles = ['Faixas','Arte','To-do','Letras','Mix','Qualidade','Exportar']
for i, t in enumerate(titles):
    tabs.set_title(i, t)

accordion = widgets.Accordion(children=[tabs])
accordion.set_title(0, 'Dashboard')

top_bar_placeholder = widgets.Box(layout=widgets.Layout(border='solid 1px #ddd', padding='6px', _dom_classes=['fixed-top-bar']))
bottom_bar_placeholder = widgets.Box(layout=widgets.Layout(border='solid 1px #ddd', padding='6px', _dom_classes=['fixed-bottom-bar']))

display(widgets.VBox([
    widgets.HTML('<div id="fixed-top-bar"><b>Controlo Rápido</b> — Este painel fica sempre visível</div>'),
    accordion,
    widgets.HTML('<div id="fixed-bottom-bar">Pré-visualizar • Métrica • Exportar • Ajuda</div>'),
]))

In [ ]:
# Tabelas com filtros/ordenação: qgrid (se disponível) ou DataFrame + filtros
try:
    import qgrid
    QGRID_AVAILABLE = True
except Exception:
    QGRID_AVAILABLE = False

def tasks_dataframe():
    proj = STATE['project']
    if not proj:
        return pd.DataFrame(columns=['title','category','status','priority','assignee','due'])
    rows = []
    for t in proj.tasks:
        rows.append({'title': t.title, 'category': t.category, 'status': t.status.value if hasattr(t.status,'value') else str(t.status), 'priority': t.priority.value if hasattr(t.priority,'value') else str(t.priority), 'assignee': t.assignee, 'due': t.due})
    return pd.DataFrame(rows)

def tracks_dataframe():
    proj = STATE['project']
    if not proj:
        return pd.DataFrame(columns=['index','title','disc','lead_single','duration_sec'])
    rows = []
    for i, tr in enumerate(proj.tracks):
        rows.append({'index': i+1, 'title': tr.title, 'disc': tr.disc, 'lead_single': tr.is_lead_single, 'duration_sec': tr.duration_sec})
    return pd.DataFrame(rows)

tasks_df = tasks_dataframe()
tracks_df = tracks_dataframe()

if QGRID_AVAILABLE:
    grid_tasks = qgrid.show_grid(tasks_df, show_toolbar=True)
    grid_tracks = qgrid.show_grid(tracks_df, show_toolbar=True)
    display(widgets.VBox([widgets.Label('Tarefas (qgrid)'), grid_tasks, widgets.Label('Faixas (qgrid)'), grid_tracks]))
else:
    # Filtros simples
    task_filter = widgets.Text(description='Filtro Tarefa:')
    track_filter = widgets.Text(description='Filtro Faixa:')
    out_tables = widgets.Output()
    def render_tables():
        with out_tables:
            clear_output()
            tdf = tasks_dataframe()
            if task_filter.value.strip():
                tdf = tdf[tdf.apply(lambda r: task_filter.value.lower() in str(r).lower(), axis=1)]
            display(Markdown('### Tarefas'))
            display(tdf)
            fdf = tracks_dataframe()
            if track_filter.value.strip():
                fdf = fdf[fdf.apply(lambda r: track_filter.value.lower() in str(r).lower(), axis=1)]
            display(Markdown('### Faixas'))
            display(fdf)
    task_filter.observe(lambda c: render_tables(), names='value')
    track_filter.observe(lambda c: render_tables(), names='value')
    render_tables()
    display(widgets.VBox([widgets.HBox([task_filter, track_filter]), out_tables]))

In [ ]:
# Métricas expandidas: crest factor, espectros médios, tonal balance
from math import log10
metrics_output = widgets.Output()

def crest_factor_metrics(y):
    if y is None or len(y)==0: return None
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    rms = float(np.sqrt(np.mean(y**2))) if y.size else 0.0
    if rms == 0:
        return {'peak': peak, 'rms': rms, 'crest_ratio': None, 'crest_db': None}
    crest_ratio = peak / rms
    crest_db = 20*log10(crest_ratio) if crest_ratio>0 else None
    return {'peak': peak, 'rms': rms, 'crest_ratio': crest_ratio, 'crest_db': crest_db}

def average_spectrum(y, sr, n_fft=4096, hop=1024):
    if not AUDIO_ANALYSIS_AVAILABLE: return None, None
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))
    spec = np.mean(S, axis=1)
    freqs = np.linspace(0, sr/2, num=len(spec))
    return freqs, spec

def tonal_balance(spec, freqs, target_curve='flat'):
    if spec is None or freqs is None: return None
    # Simplificação: comparar bandas (low, mid, high) com alvo flat
    low = np.mean(spec[(freqs>=20)&(freqs<200)]) if np.any((freqs>=20)&(freqs<200)) else 0
    mid = np.mean(spec[(freqs>=200)&(freqs<2000)]) if np.any((freqs>=200)&(freqs<2000)) else 0
    high = np.mean(spec[(freqs>=2000)&(freqs<=20000)]) if np.any((freqs>=2000)&(freqs<=20000)) else 0
    total = low+mid+high if (low+mid+high)>0 else 1
    return {'low': low/total, 'mid': mid/total, 'high': high/total}

def compute_metrics_for_selected_mix():
    with metrics_output:
        clear_output()
        if not AUDIO_ANALYSIS_AVAILABLE:
            display(Markdown('Librosa indisponível. Instale para analisar áudio.'))
            return
        proj = STATE['project']
        if not proj: return
        idx = mix_selector.value
        if idx is None: return
        t = proj.tracks[idx]
        if not t.mixes:
            display(Markdown('Sem versões de mix.'))
            return
        # Use até 2 versões p/ comparação A/B
        versions = t.mixes[:2]
        comp = []
        for mv in versions:
            if not mv.path or not Path(mv.path).exists():
                continue
            y, sr = librosa.load(mv.path, sr=None, mono=True)
            cf = crest_factor_metrics(y)
            f, s = average_spectrum(y, sr)
            tb = tonal_balance(s, f)
            comp.append({'name': mv.name, 'cf': cf, 'freqs': f, 'spec': s, 'tb': tb})
        if not comp:
            display(Markdown('Não foi possível analisar mixes.'))
            return
        for c in comp:
            cf = c['cf'] or {}
            display(Markdown(f"### {c['name']} — Crest: {cf.get('crest_db','?')} dB (peak={cf.get('peak','?'):.3f} / rms={cf.get('rms','?'):.3f})"))
        if PLOTLY_AVAILABLE and len(comp)>=1 and comp[0]['spec'] is not None:
            fig = go.Figure()
            for c in comp:
                if c['spec'] is not None:
                    fig.add_scatter(x=c['freqs'], y=20*np.log10(c['spec']+1e-9), mode='lines', name=c['name'])
            fig.update_layout(title='Espectro médio (dB)', xaxis_type='log', xaxis_title='Frequência (Hz)', yaxis_title='Magnitude (dB)')
            fig.show()
        if len(comp)>=1 and comp[0]['tb'] is not None:
            tb_fig = go.Figure(data=[go.Bar(x=['Low','Mid','High'], y=[comp[0]['tb']['low'], comp[0]['tb']['mid'], comp[0]['tb']['high']])]) if PLOTLY_AVAILABLE else None
            if tb_fig: tb_fig.update_layout(title=f"Tonal balance — {comp[0]['name']}")
            if tb_fig: tb_fig.show()

metrics_btn = widgets.Button(description='Calcular Métricas', icon='bar-chart')
metrics_btn.on_click(lambda b: compute_metrics_for_selected_mix())
display(widgets.VBox([widgets.Label('Métricas Expandidas'), widgets.HBox([mix_selector, metrics_btn]), metrics_output]))

In [ ]:
# Coletânea a partir de singles: combinar projetos preservando dados
from typing import Tuple
compile_output = widgets.Output()
singles_ms = widgets.SelectMultiple(options=[], description='Singles:')
comp_name = widgets.Text(description='Coletânea:', placeholder='Nome da coletânea')
comp_fmt = widgets.Dropdown(options=[ProjectFormat.MIXTAPE, ProjectFormat.LP], description='Formato:')
copy_audio_chk = widgets.Checkbox(value=False, description='Copiar arquivos de áudio (em vez de referenciar)')
make_comp_btn = widgets.Button(description='Criar Coletânea', icon='layers')

def list_singles_only() -> List[str]:
    slugs = list_projects()
    out = []
    for s in slugs:
        p = load_project(s)
        if p and p.fmt == ProjectFormat.SINGLE:
            out.append(s)
    return sorted(out)

def create_compilation(slugs: List[str], name: str, fmt: ProjectFormat, copy_audio: bool) -> Optional[Project]:
    if not slugs: return None
    comp = Project(name=name or 'Coletânea', artist='', fmt=fmt)
    for slug in slugs:
        p = load_project(slug)
        if not p: continue
        for t in p.tracks:
            new_t = Track(title=t.title, artist=p.artist or comp.artist, is_lead_single=t.is_lead_single, disc=t.disc, lyrics=t.lyrics)
            # Referenciar ou copiar áudio
            if t.audio_path and Path(t.audio_path).exists():
                if copy_audio:
                    dest = project_root(comp) / 'tracks' / Path(t.audio_path).name
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(t.audio_path, dest)
                    new_t.audio_path = str(dest)
                else:
                    new_t.audio_path = t.audio_path
            comp.tracks.append(new_t)
    save_project(comp)
    return comp

def refresh_singles_list():
    singles_ms.options = list_singles_only()
refresh_singles_list()

def handle_make_comp(_):
    slugs = list(singles_ms.value)
    comp = create_compilation(slugs, comp_name.value.strip(), comp_fmt.value, copy_audio_chk.value)
    with compile_output:
        clear_output()
        if comp:
            display(Markdown(f"Criada coletânea: **{comp.name}** com {len(comp.tracks)} faixas"))
            existing_dd.options = ['<novo>'] + list_projects()
        else:
            display(Markdown("Falha ao criar coletânea"))

make_comp_btn.on_click(handle_make_comp)
display(widgets.VBox([widgets.Label('Coletânea de Singles'), widgets.HBox([singles_ms, widgets.VBox([comp_name, comp_fmt, copy_audio_chk, make_comp_btn])]), compile_output]))

In [ ]:
# Venues DB: persistência + UI (Processo 2 – Venues)
import json, requests
from dataclasses import dataclass, asdict
from typing import List, Optional

VENUES_PATH = BASE_DIR / '_venues.json'
VENUE_PHOTOS_DIR = BASE_DIR / '_venue_photos'
VENUE_PHOTOS_DIR.mkdir(exist_ok=True)

@dataclass
class Venue:
    name: str
    address: str = ''
    city: str = ''
    country: str = ''
    capacity: Optional[int] = None
    email: str = ''
    phone: str = ''
    photos: List[str] = None
    notes: str = ''

def load_venues() -> List[Venue]:
    if VENUES_PATH.exists():
        with open(VENUES_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return [Venue(**v) for v in data]
    return []

def save_venues(venues: List[Venue]):
    with open(VENUES_PATH, 'w', encoding='utf-8') as f:
        json.dump([asdict(v) for v in venues], f, ensure_ascii=False, indent=2)

venues_state = load_venues()

# UI
venue_list = widgets.Select(options=[v.name for v in venues_state], description='Venues:')
new_name = widgets.Text(description='Nome:')
new_addr = widgets.Text(description='Endereço:')
new_city = widgets.Text(description='Cidade:')
new_country = widgets.Text(description='País:')
new_capacity = widgets.BoundedIntText(description='Capacidade:', min=0, max=100000, value=0)
new_email = widgets.Text(description='Email:')
new_phone = widgets.Text(description='Telefone:')
new_notes = widgets.Textarea(description='Notas:')
photo_url = widgets.Text(description='Foto URL:')
add_photo_btn = widgets.Button(description='Adicionar Foto', icon='image')
add_venue_btn = widgets.Button(description='Adicionar novo venue', icon='plus', button_style='success')
update_venue_btn = widgets.Button(description='Atualizar', icon='save')
delete_venue_btn = widgets.Button(description='Remover', icon='trash', button_style='danger')
venue_output = widgets.Output()

def refresh_venue_list():
    venue_list.options = [v.name for v in venues_state]
    if venues_state:
        venue_list.value = venues_state[0].name

def current_venue() -> Optional[Venue]:
    if not venue_list.value: return None
    for v in venues_state:
        if v.name == venue_list.value: return v
    return None

def load_form(v: Venue):
    new_name.value = v.name
    new_addr.value = v.address
    new_city.value = v.city
    new_country.value = v.country
    new_capacity.value = v.capacity or 0
    new_email.value = v.email
    new_phone.value = v.phone
    new_notes.value = v.notes

def handle_select(change):
    v = current_venue()
    if v: load_form(v)

def handle_add(_):
    v = Venue(name=new_name.value.strip() or 'Novo Venue',
              address=new_addr.value, city=new_city.value, country=new_country.value,
              capacity=int(new_capacity.value or 0), email=new_email.value, phone=new_phone.value, notes=new_notes.value, photos=[])
    venues_state.append(v)
    save_venues(venues_state)
    refresh_venue_list()
    with venue_output:
        clear_output()
        display(Markdown(f"Adicionado: **{v.name}**"))

def handle_update(_):
    v = current_venue()
    if not v: return
    v.name = new_name.value.strip() or v.name
    v.address = new_addr.value
    v.city = new_city.value
    v.country = new_country.value
    v.capacity = int(new_capacity.value or 0)
    v.email = new_email.value
    v.phone = new_phone.value
    v.notes = new_notes.value
    save_venues(venues_state)
    refresh_venue_list()
    with venue_output:
        clear_output()
        display(Markdown(f"Atualizado: **{v.name}**"))

def handle_delete(_):
    v = current_venue()
    if not v: return
    venues_state[:] = [x for x in venues_state if x.name != v.name]
    save_venues(venues_state)
    refresh_venue_list()
    with venue_output:
        clear_output()
        display(Markdown("Venue removido."))

def handle_add_photo(_):
    v = current_venue()
    if not v:
        with venue_output:
            clear_output()
            display(Markdown('Selecione ou crie um venue para adicionar foto.'))
        return
    url = photo_url.value.strip()
    if not url: return
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        ext = '.jpg'
        fname = slugify(v.name) + '-' + str(int(time.time())) + ext
        path = VENUE_PHOTOS_DIR / fname
        with open(path, 'wb') as f: f.write(r.content)
        if v.photos is None: v.photos = []
        v.photos.append(str(path))
        save_venues(venues_state)
        with venue_output:
            clear_output()
            display(Markdown(f"Foto adicionada: `{path}`"))
    except Exception as e:
        with venue_output:
            clear_output()
            display(Markdown(f"Falha ao baixar imagem: {e}"))

venue_list.observe(handle_select, names='value')
add_venue_btn.on_click(handle_add)
update_venue_btn.on_click(handle_update)
delete_venue_btn.on_click(handle_delete)
add_photo_btn.on_click(handle_add_photo)

refresh_venue_list()
display(Markdown('## Processo 2 – Venues'))
display(widgets.VBox([widgets.HBox([venue_list, add_venue_btn, delete_venue_btn]), new_name, new_addr, widgets.HBox([new_city, new_country, new_capacity]), widgets.HBox([new_email, new_phone]), new_notes, widgets.HBox([photo_url, add_photo_btn]), update_venue_btn, venue_output]))

In [ ]:
# Vestuário DB: histórico + tooltips em PT e extras
WARDROBE_PATH = BASE_DIR / '_wardrobe.json'

from dataclasses import dataclass, asdict
from typing import Dict

@dataclass
class WardrobeItem:
    name: str
    category: str
    description_pt: str = ''
    period: str = ''
    refs: List[str] = None
    images: List[str] = None

def load_wardrobe() -> List[WardrobeItem]:
    if WARDROBE_PATH.exists():
        with open(WARDROBE_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return [WardrobeItem(**w) for w in data]
    return []

def save_wardrobe(items: List[WardrobeItem]):
    with open(WARDROBE_PATH, 'w', encoding='utf-8') as f:
        json.dump([asdict(w) for w in items], f, ensure_ascii=False, indent=2)

wardrobe_state = load_wardrobe()

# Preseed items (run once idempotent)
preseed = [
    WardrobeItem(name='Camicia', category='Base', period='Itália (c.1420-1480)', description_pt='Camisa de linho, mangas estreitas; gola alta em fita com dobra frontal.'),
    WardrobeItem(name='Mutande', category='Base', period='Itália (c.1420-1480)', description_pt='Cueca curta e ajustada, estilo briefs.'),
    WardrobeItem(name='Brache', category='Base', period='Itália (c.1420-1480)', description_pt='Cueca mais solta, estilo boxer, com forro de linho no topo.'),
    WardrobeItem(name='Farsetto (Doublet)', category='Camada Média', period='Itália (c.1420-1480)', description_pt='Jaqueta acolchoada justa, gola alta; prende as calze por ilhós.'),
    WardrobeItem(name='Calze (Hose)', category='Perna/Meia', period='Itália (c.1420-1480)', description_pt='Meias compridas separadas, cortadas no viés, forradas em linho no topo.'),
    WardrobeItem(name='Giornea/Cioppa/Gonella', category='Sobrecamada', period='Itália (c.1420-1480)', description_pt='Sobrevestes com pregas arredondadas; muitas vezes forradas em pele.'),
    WardrobeItem(name='Mantello', category='Capa', period='Itália (c.1420-1480)', description_pt='Capa longa com pequena gola; usualmente vermelha ou preta.'),
    WardrobeItem(name='Coroa', category='Acessório', period='Vários', description_pt='Coroa ornamental, simbólica, usada em performances especiais.'),
    WardrobeItem(name='Aketon (Viking)', category='Proteção', period='Medieval', description_pt='Gibão acolchoado para amortecer impactos; versão leve para mobilidade.'),
    WardrobeItem(name='Batina de seda', category='Sobrecamada', period='Renascentista', description_pt='Veste longa de seda, usada em contextos formais e litúrgicos.'),
    WardrobeItem(name='Leather Arm Garter', category='Acessório', period='Medieval/Renasc.', description_pt='Cinta de couro para ajustar mangas/antebraço, estética e função.'),
    WardrobeItem(name='Arming Gambeson (Doublet HEMA)', category='Proteção', period='Tardio', description_pt='Gambeson acolchoado para treino/combate com dupla função de doublet.'),
    WardrobeItem(name='Gambeson leve (não muito fofo)', category='Proteção', period='Medieval/Renasc.', description_pt='Versão menos volumosa para mobilidade e estética equilibradas.'),
]
names = {w.name for w in wardrobe_state}
for w in preseed:
    if w.name not in names: wardrobe_state.append(w)
if preseed: save_wardrobe(wardrobe_state)

# UI
w_category = widgets.Dropdown(options=['Base','Camada Média','Perna/Meia','Sobrecamada','Capa','Proteção','Acessório','Outros'], description='Categoria:')
w_name = widgets.Text(description='Nome:')
w_desc = widgets.Textarea(description='Descrição PT:')
w_period = widgets.Text(description='Período:')
w_ref = widgets.Text(description='Ref URL:')
w_add_ref_btn = widgets.Button(description='Add Ref', icon='plus')
w_add_btn = widgets.Button(description='Adicionar Peça', icon='plus', button_style='success')
w_list = widgets.Select(options=[f"{w.category}: {w.name}" for w in wardrobe_state], description='Peças:')
w_tooltip = widgets.HTML('Passe o rato sobre uma peça para ver a descrição.')
w_output = widgets.Output()

def refresh_wlist():
    w_list.options = [f"{w.category}: {w.name}" for w in wardrobe_state]

def find_item_from_option(opt: str) -> Optional[WardrobeItem]:
    try: cat, name = opt.split(': ', 1)
    except: return None
    for w in wardrobe_state:
        if w.name == name and w.category == cat: return w
    return None

def on_select_item(change):
    item = find_item_from_option(change['new']) if change['new'] else None
    if item:
        w_category.value = item.category if item.category in [o for o in w_category.options] else 'Outros'
        w_name.value = item.name
        w_desc.value = item.description_pt
        w_period.value = item.period
        # Tooltip com descrição
        w_tooltip.value = f"<b>{item.name}</b>: {item.description_pt}"

def add_ref(_):
    item = find_item_from_option(w_list.value) if w_list.value else None
    if not item:
        with w_output:
            clear_output()
            display(Markdown('Selecione uma peça para anexar referência.'))
        return
    url = w_ref.value.strip()
    if not url: return
    if item.refs is None: item.refs = []
    item.refs.append(url)
    save_wardrobe(wardrobe_state)
    with w_output:
        clear_output()
        display(Markdown('Referência adicionada.'))

def add_item(_):
    item = WardrobeItem(name=w_name.value.strip() or 'Nova Peça', category=w_category.value, description_pt=w_desc.value, period=w_period.value, refs=[], images=[])
    wardrobe_state.append(item)
    save_wardrobe(wardrobe_state)
    refresh_wlist()
    with w_output:
        clear_output()
        display(Markdown(f"Peça adicionada: **{item.name}**"))

w_list.observe(on_select_item, names='value')
w_add_ref_btn.on_click(add_ref)
w_add_btn.on_click(add_item)
refresh_wlist()
display(Markdown('## Vestuário — Base histórica italiana (c.1420-1480) + extras'))
display(widgets.VBox([widgets.HBox([w_category, w_name]), w_desc, widgets.HBox([w_period, w_ref, w_add_ref_btn]), w_add_btn, w_tooltip, w_list, w_output]))

In [ ]:
# Email: envio via SMTP (usar vars de ambiente)
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

SMTP_HOST = os.getenv('SMTP_HOST','')
SMTP_PORT = int(os.getenv('SMTP_PORT','587') or 587)
SMTP_USER = os.getenv('SMTP_USER','')
SMTP_PASS = os.getenv('SMTP_PASS','')
SMTP_TLS = os.getenv('SMTP_TLS','true').lower() in ('1','true','yes')

to_tb = widgets.Text(description='Para:')
cc_tb = widgets.Text(description='Cc:')
subj_tb = widgets.Text(description='Assunto:')
template_ta = widgets.Textarea(description='Corpo:', layout=widgets.Layout(width='100%', height='160px'))
send_btn = widgets.Button(description='Enviar Email', icon='paper-plane', button_style='primary')
email_out = widgets.Output()

def fill_template(context: dict, template: str) -> str:
    try:
        return template.format(**context)
    except Exception:
        return template

def send_email_smtp(to_addrs: List[str], cc_addrs: List[str], subject: str, body: str):
    msg = MIMEMultipart()
    msg['From'] = SMTP_USER
    msg['To'] = ', '.join(to_addrs)
    msg['Cc'] = ', '.join(cc_addrs) if cc_addrs else ''
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain', 'utf-8'))
    recipients = to_addrs + cc_addrs

    if SMTP_TLS:
        context = ssl.create_default_context()
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.starttls(context=context)
            if SMTP_USER: server.login(SMTP_USER, SMTP_PASS)
            server.sendmail(SMTP_USER or 'noreply@example.com', recipients, msg.as_string())
    else:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            if SMTP_USER: server.login(SMTP_USER, SMTP_PASS)
            server.sendmail(SMTP_USER or 'noreply@example.com', recipients, msg.as_string())

def handle_send(_):
    if not SMTP_HOST:
        with email_out:
            clear_output()
            display(Markdown('Configure SMTP_HOST/PORT/USER/PASS nas variáveis de ambiente.'))
        return
    # Contexto autopreenchido (venue/projeto)
    ctx = {}
    v = current_venue()
    if v:
        ctx.update({'venue_name': v.name, 'venue_city': v.city, 'venue_country': v.country, 'venue_email': v.email})
    proj = STATE.get('project')
    if proj:
        ctx.update({'project_name': proj.name, 'project_artist': proj.artist, 'project_fmt': proj.fmt.value if hasattr(proj.fmt,'value') else str(proj.fmt)})
    body = fill_template(ctx, template_ta.value)
    to_addrs = [x.strip() for x in to_tb.value.split(',') if x.strip()] or ([v.email] if v and v.email else [])
    cc_addrs = [x.strip() for x in cc_tb.value.split(',') if x.strip()]
    try:
        send_email_smtp(to_addrs, cc_addrs, subj_tb.value, body)
        with email_out:
            clear_output()
            display(Markdown('Email enviado.'))
    except Exception as e:
        with email_out:
            clear_output()
            display(Markdown(f'Falha ao enviar email: {e}'))

send_btn.on_click(handle_send)
display(Markdown('## Email — Templates com autofill (venue/projeto)'))
display(widgets.VBox([widgets.HBox([to_tb, cc_tb]), subj_tb, template_ta, send_btn, email_out]))

## Tour planning (Master Tour–like)

Plan an itinerary of shows across your Venues database with dates/times and optional routing. If a HERE API key is available (env var HERE_API_KEY), the dashboard will estimate driving distance and duration between consecutive stops. Export the schedule to CSV or iCalendar (.ics).

In [ ]:
# Tour planning module: models, routing helpers, UI, and exports
from pathlib import Path
import os, json, math
import datetime as dt
from typing import List, Dict, Any, Optional

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as e:
    widgets = None
    print("ipywidgets not available:", e)

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import requests
except Exception:
    requests = None

PROJECTS_ROOT = Path("Projects")
PROJECTS_ROOT.mkdir(parents=True, exist_ok=True)
VENUES_DB = PROJECTS_ROOT / "_venues.json"

# --- Venues loading ---

def _load_json(path: Path, default):
    try:
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Failed to read {path}: {e}")
    return default

def load_venues() -> Dict[str, Dict[str, Any]]:
    """Return a dict keyed by venue id or name with at least name, city, country, lat, lon (optional)."""
    data = _load_json(VENUES_DB, {})
    # Normalize: allow file to be list or dict
    venues: Dict[str, Dict[str, Any]] = {}
    if isinstance(data, list):
        for v in data:
            key = str(v.get("id") or v.get("name") or f"venue_{len(venues)+1}")
            venues[key] = v
    elif isinstance(data, dict):
        # Either {id: {...}} or {"venues": [...]} shape
        if "venues" in data and isinstance(data["venues"], list):
            for v in data["venues"]:
                key = str(v.get("id") or v.get("name") or f"venue_{len(venues)+1}")
                venues[key] = v
        else:
            for k, v in data.items():
                venues[str(k)] = v if isinstance(v, dict) else {"name": str(v)}
    return venues

# --- Routing helpers ---

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def here_route_summary(origin: Dict[str, float], dest: Dict[str, float], departure: Optional[dt.datetime] = None) -> Optional[Dict[str, float]]:
    """Returns {'distance_km': float, 'duration_min': float} using HERE Routing if available, else None."""
    api_key = os.getenv("HERE_API_KEY")
    if not (api_key and requests):
        return None
    try:
        params = {
            "transportMode": "car",
            "origin": f"{origin['lat']},{origin['lon']}",
            "destination": f"{dest['lat']},{dest['lon']}",
            "return": "summary",
            "apiKey": api_key,
        }
        if departure:
            # Use ISO8601
            params["departureTime"] = departure.isoformat(timespec="seconds")
        url = "https://router.hereapi.com/v8/routes"
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
        route = data.get("routes", [{}])[0]
        sections = route.get("sections", [{}])
        summary = sections[0].get("summary", {}) if sections else {}
        length_m = float(summary.get("length", 0))
        duration_s = float(summary.get("duration", 0))
        if length_m <= 0 or duration_s <= 0:
            return None
        return {"distance_km": length_m/1000.0, "duration_min": duration_s/60.0}
    except Exception as e:
        print("HERE routing failed:", e)
        return None

# --- Itinerary persistence ---

def plan_path(slug: str) -> Path:
    return PROJECTS_ROOT / slug / "tour.json"

def load_plan(slug: str) -> List[Dict[str, Any]]:
    p = plan_path(slug)
    return _load_json(p, [])

def save_plan(slug: str, plan: List[Dict[str, Any]]):
    p = plan_path(slug)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding="utf-8")

# --- Utilities ---

def list_projects() -> List[str]:
    slugs = []
    for d in PROJECTS_ROOT.iterdir():
        if d.is_dir() and (d/"project.json").exists():
            slugs.append(d.name)
    return sorted(slugs)

TIME_FIELDS = ["load_in", "soundcheck", "doors", "set_time", "curfew"]

def _parse_hhmm(s: str) -> Optional[dt.time]:
    try:
        hh, mm = s.split(":")
        return dt.time(int(hh), int(mm))
    except Exception:
        return None

# --- UI ---
if widgets is not None:
    venues = load_venues()
    venue_options = []
    for key, v in venues.items():
        name = v.get("name", key)
        city = v.get("city") or v.get("cidade") or ""
        country = v.get("country") or v.get("pais") or ""
        label = f"{name} — {city} {country}".strip()
        venue_options.append((label, key))
    if not venue_options:
        venue_options = [("(Nenhum venue no banco — adicione em Venues)", "__none__")]

    project_dd = widgets.Dropdown(options=[("Selecione um projeto", "__none__")] + [(s, s) for s in list_projects()], description="Projeto:")
    venue_dd = widgets.Dropdown(options=venue_options, description="Venue:")
    date_dp = widgets.DatePicker(description="Data:")
    load_in = widgets.Text(placeholder="HH:MM", description="Load-in:")
    soundcheck = widgets.Text(placeholder="HH:MM", description="Soundcheck:")
    doors = widgets.Text(placeholder="HH:MM", description="Portas:")
    set_time = widgets.Text(placeholder="HH:MM", description="Show:")
    curfew = widgets.Text(placeholder="HH:MM", description="Curfew:")

    compute_travel_chk = widgets.Checkbox(value=True, description="Estimar viagem do show anterior")
    travel_out = widgets.HTML(value="<i>Distância/duração serão calculadas quando possível.</i>")

    add_btn = widgets.Button(description="Adicionar parada", button_style="success")
    save_btn = widgets.Button(description="Guardar plano", button_style="info")
    export_csv_btn = widgets.Button(description="Exportar CSV")
    export_ics_btn = widgets.Button(description="Exportar iCalendar (.ics)")

    table_out = widgets.Output()
    msg_out = widgets.Output()

    current_plan: List[Dict[str, Any]] = []

    def refresh_table():
        table_out.clear_output()
        with table_out:
            if not current_plan:
                print("Nenhuma parada ainda.")
            else:
                if pd is not None:
                    df = pd.DataFrame(current_plan)
                    display(df)
                else:
                    for i, stop in enumerate(current_plan, 1):
                        print(i, stop)

    def on_add_clicked(_):
        if project_dd.value in (None, "__none__"):
            with msg_out:
                clear_output()
                print("Selecione um projeto primeiro.")
            return
        if venue_dd.value in (None, "__none__"):
            with msg_out:
                clear_output()
                print("Selecione um venue válido.")
            return
        v = venues.get(venue_dd.value, {})
        stop = {
            "venue_key": venue_dd.value,
            "venue_name": v.get("name", venue_dd.value),
            "city": v.get("city") or v.get("cidade"),
            "country": v.get("country") or v.get("pais"),
            "date": date_dp.value.isoformat() if date_dp.value else None,
        }
        for f, w in [("load_in", load_in), ("soundcheck", soundcheck), ("doors", doors), ("set_time", set_time), ("curfew", curfew)]:
            stop[f] = w.value or None
        # compute travel from previous stop if possible
        if compute_travel_chk.value and len(current_plan) >= 1:
            prev = current_plan[-1]
            def latlon(x):
                lat = x.get("lat") or x.get("latitude")
                lon = x.get("lon") or x.get("longitude")
                return lat, lon
            plat, plon = latlon(venues.get(prev["venue_key"], {}))
            clat, clon = latlon(v)
            if plat is not None and plon is not None and clat is not None and clon is not None:
                # Try HERE first
                summ = here_route_summary({"lat": plat, "lon": plon}, {"lat": clat, "lon": clon}, None)
                if summ:
                    stop["distance_km"] = round(summ["distance_km"], 1)
                    stop["drive_min"] = int(round(summ["duration_min"]))
                    travel_out.value = f"<b>Viagem:</b> ~{stop['distance_km']} km, {stop['drive_min']} min (HERE)"
                else:
                    dist = haversine_km(plat, plon, clat, clon)
                    stop["distance_km"] = round(dist, 1)
                    # Assume avg 80 km/h
                    stop["drive_min"] = int(round((dist/80.0)*60))
                    travel_out.value = f"<b>Viagem (aprox.):</b> ~{stop['distance_km']} km, {stop['drive_min']} min"
            else:
                travel_out.value = "<i>Sem coordenadas nos venues — adicione lat/lon no módulo Venues para estimativas.</i>"
        current_plan.append(stop)
        refresh_table()
        with msg_out:
            clear_output()
            print("Parada adicionada.")

    def on_save_clicked(_):
        if project_dd.value in (None, "__none__"):
            with msg_out:
                clear_output()
                print("Selecione um projeto primeiro.")
            return
        save_plan(project_dd.value, current_plan)
        with msg_out:
            clear_output()
            print("Plano salvo.")

    def export_csv(_):
        if project_dd.value in (None, "__none__"):
            with msg_out:
                clear_output()
                print("Selecione um projeto primeiro.")
            return
        import csv
        out_dir = PROJECTS_ROOT / project_dd.value / "exports"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_csv = out_dir / f"{project_dd.value}-tour.csv"
        cols = ["date", "venue_name", "city", "country"] + TIME_FIELDS + ["distance_km", "drive_min"]
        with out_csv.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for row in current_plan:
                w.writerow({k: row.get(k) for k in cols})
        with msg_out:
            clear_output()
            print("CSV exportado:", out_csv)

    def export_ics(_):
        if project_dd.value in (None, "__none__"):
            with msg_out:
                clear_output()
                print("Selecione um projeto primeiro.")
            return
        out_dir = PROJECTS_ROOT / project_dd.value / "exports"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_ics = out_dir / f"{project_dd.value}-tour.ics"
        lines = [
            "BEGIN:VCALENDAR",
            "VERSION:2.0",
            "PRODID:-//pretos//tour//PT",
        ]
        for row in current_plan:
            d = row.get("date")
            if not d:
                continue
            # Compose DTSTART/DTEND from date + set_time/curfew
            try:
                date_obj = dt.date.fromisoformat(d)
            except Exception:
                continue
            def to_dtstr(t: Optional[str], default_h=18, default_m=0):
                if t:
                    tm = _parse_hhmm(t)
                    if tm:
                        return dt.datetime.combine(date_obj, tm).strftime("%Y%m%dT%H%M%S")
                return dt.datetime.combine(date_obj, dt.time(default_h, default_m)).strftime("%Y%m%dT%H%M%S")
            dtstart = to_dtstr(row.get("set_time"), 20, 0)
            dtend = to_dtstr(row.get("curfew"), 22, 0)
            summary = f"Show: {row.get('venue_name','')}"
            location = ", ".join([x for x in [row.get("venue_name"), row.get("city"), row.get("country")] if x])
            desc_parts = []
            for f in TIME_FIELDS:
                if row.get(f):
                    desc_parts.append(f"{f}: {row[f]}")
            if row.get("distance_km"):
                desc_parts.append(f"viagem: {row['distance_km']} km")
            if row.get("drive_min"):
                desc_parts.append(f"{row['drive_min']} min")
            description = "\\n".join(desc_parts)
            lines += [
                "BEGIN:VEVENT",
                f"DTSTART:{dtstart}",
                f"DTEND:{dtend}",
                f"SUMMARY:{summary}",
                f"LOCATION:{location}",
                f"DESCRIPTION:{description}",
                "END:VEVENT",
            ]
        lines += ["END:VCALENDAR"]
        out_ics.write_text("\n".join(lines), encoding="utf-8")
        with msg_out:
            clear_output()
            print("iCalendar exportado:", out_ics)

    add_btn.on_click(on_add_clicked)
    save_btn.on_click(on_save_clicked)
    export_csv_btn.on_click(export_csv)
    export_ics_btn.on_click(export_ics)

    # Load existing plan when project changes
    def on_project_change(change):
        if change.get("name") == "value":
            slug = change.get("new")
            if slug in (None, "__none__"):
                return
            nonlocal_plan = load_plan(slug)
            current_plan.clear()
            current_plan.extend(nonlocal_plan)
            refresh_table()
            with msg_out:
                clear_output()
                print(f"Projeto '{slug}' carregado. Paradas: {len(current_plan)}")
    project_dd.observe(on_project_change)

    ui = widgets.VBox([
        widgets.HTML("<b>Planeamento de tour</b>"),
        project_dd,
        widgets.HBox([venue_dd, date_dp]),
        widgets.HBox([load_in, soundcheck, doors, set_time, curfew]),
        widgets.HBox([compute_travel_chk, travel_out]),
        widgets.HBox([add_btn, save_btn, export_csv_btn, export_ics_btn]),
        table_out,
        msg_out,
    ])
    display(ui)
else:
    print("ipywidgets indisponível — não é possível renderizar o UI de tour.")


## Export to PDF (nbconvert/pandoc fallback)

Generate a lightweight project report in Markdown and attempt to convert it to PDF using pypandoc (Pandoc). If conversion tools aren't installed, the Markdown and HTML files will be saved as a fallback.

In [ ]:
# PDF export utilities and UI
from pathlib import Path
import json

try:
    import pypandoc
except Exception:
    pypandoc = None

try:
    from markdown import markdown as md_to_html
except Exception:
    md_to_html = None

try:
    from weasyprint import HTML as WeasyHTML
except Exception:
    WeasyHTML = None

PROJECTS_ROOT = Path("Projects")


def _read_project(slug: str) -> dict:
    p = PROJECTS_ROOT / slug / "project.json"
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return {"name": slug}
    return {"name": slug}


def _read_tour(slug: str):
    t = PROJECTS_ROOT / slug / "tour.json"
    if t.exists():
        try:
            return json.loads(t.read_text(encoding="utf-8"))
        except Exception:
            return []
    return []


def project_markdown(slug: str) -> str:
    proj = _read_project(slug)
    name = proj.get("name") or proj.get("title") or slug
    lines = [f"# Projeto: {name}", ""]
    # Basic overview
    lines += ["## Resumo", "- Slug: ``{}``".format(slug)]
    # Tracks (if present)
    tracks = proj.get("tracks") or []
    if isinstance(tracks, list) and tracks:
        lines += ["", "## Faixas", ""]
        for i, tr in enumerate(tracks, 1):
            title = tr.get("title") or tr.get("name") or f"Track {i}"
            duration = tr.get("duration") or tr.get("length")
            if duration:
                lines.append(f"- {i}. {title} — {duration}")
            else:
                lines.append(f"- {i}. {title}")
    # Tour section if any
    tour = _read_tour(slug)
    if isinstance(tour, list) and tour:
        lines += ["", "## Tour", "", "| Data | Venue | Cidade | País | Show | Curfew | Distância | Drive |", "|---|---|---|---|---|---|---:|---:|"]
        for row in tour:
            lines.append("| {} | {} | {} | {} | {} | {} | {} km | {} min |".format(
                (row.get("date") or "").split("T")[0],
                row.get("venue_name", ""),
                row.get("city", ""),
                row.get("country", ""),
                row.get("set_time", ""),
                row.get("curfew", ""),
                row.get("distance_km", ""),
                row.get("drive_min", ""),
            ))
    lines.append("")
    return "\n".join(lines)


def export_pdf_from_markdown(md_text: str, out_pdf: Path) -> Path:
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    # Try pypandoc first
    if pypandoc is not None:
        try:
            pypandoc.convert_text(md_text, to="pdf", format="md", outputfile=str(out_pdf))
            return out_pdf
        except Exception as e:
            print("pypandoc failed:", e)
    # Try via HTML -> WeasyPrint
    if md_to_html is not None and WeasyHTML is not None:
        try:
            html = md_to_html(md_text)
            WeasyHTML(string=html).write_pdf(str(out_pdf))
            return out_pdf
        except Exception as e:
            print("WeasyPrint failed:", e)
    # Fallback: save MD and HTML, notify user
    out_md = out_pdf.with_suffix(".md")
    out_html = out_pdf.with_suffix(".html")
    out_md.write_text(md_text, encoding="utf-8")
    if md_to_html is not None:
        out_html.write_text(md_to_html(md_text), encoding="utf-8")
    print("PDF conversion tools not available. Saved:", out_md, "and", out_html)
    return out_pdf


if widgets is not None:
    proj_dd = widgets.Dropdown(options=[("Selecione um projeto", "__none__")] + [(s, s) for s in sorted([d.name for d in PROJECTS_ROOT.iterdir() if (d/"project.json").exists()])], description="Projeto:")
    export_btn = widgets.Button(description="Exportar PDF", button_style="primary")
    out = widgets.Output()

    def on_export(_):
        out.clear_output()
        if proj_dd.value in (None, "__none__"):
            with out:
                print("Selecione um projeto primeiro.")
            return
        slug = proj_dd.value
        md_text = project_markdown(slug)
        pdf_path = PROJECTS_ROOT / slug / "exports" / f"{slug}-report.pdf"
        res = export_pdf_from_markdown(md_text, pdf_path)
        with out:
            if res.exists() and res.stat().st_size > 0:
                print("Relatório PDF gerado:", res)
            else:
                print("Não foi possível gerar PDF. Confira arquivos .md/.html nas exports.")

    export_btn.on_click(on_export)
    display(widgets.VBox([proj_dd, export_btn, out]))
else:
    print("ipywidgets indisponível — export UI não renderizado.")


## Sync projects from web app or other folders

Scan a set of external folders for projects and import them into this dashboard's Projects/ directory by symlink (preferred) or copy. You can configure additional roots via the SYNC_ROOTS environment variable (colon-separated).

In [ ]:
# Webapp sync: scan/import external projects
from pathlib import Path
import os, json, shutil

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    widgets = None

PROJECTS_ROOT = Path("Projects")

# Default roots + env override
SYNC_ROOTS = [
    Path("../Projects"),
    Path("Projects_sync"),
    Path("src/app/python/Projects"),
    Path("public/Projects"),
]
_env_roots = os.getenv("SYNC_ROOTS")
if _env_roots:
    for p in _env_roots.split(":" ):
        if p.strip():
            SYNC_ROOTS.append(Path(p.strip()))


def scan_external_projects():
    found = []
    seen = set()
    for root in SYNC_ROOTS:
        if not root.exists():
            continue
        for d in root.iterdir():
            if d.is_dir() and (d/"project.json").exists():
                slug = d.name
                if slug in seen:
                    continue
                seen.add(slug)
                found.append((slug, d))
    return sorted(found, key=lambda x: x[0])


def import_project(slug: str, src: Path, mode: str = "symlink") -> str:
    dst = PROJECTS_ROOT / slug
    if dst.exists():
        return f"SKIP {slug}: já existe em {dst}"
    try:
        if mode == "symlink":
            os.symlink(src, dst)
            return f"LINK {slug}: {dst} -> {src}"
        else:
            shutil.copytree(src, dst)
            return f"COPY {slug}: {src} -> {dst}"
    except Exception as e:
        if mode == "symlink":
            # Fallback to copy
            try:
                shutil.copytree(src, dst)
                return f"COPY (fallback) {slug}: {src} -> {dst}"
            except Exception as e2:
                return f"ERRO {slug}: {e2}"
        return f"ERRO {slug}: {e}"


if widgets is not None:
    refresh_btn = widgets.Button(description="Re-escanear", button_style="info")
    mode_rb = widgets.RadioButtons(options=[("Ligação (symlink)", "symlink"), ("Copiar", "copy")], value="symlink", description="Modo:")
    items_select = widgets.SelectMultiple(options=[], description="Projetos:", rows=8)
    import_btn = widgets.Button(description="Importar selecionados", button_style="success")
    out = widgets.Output()

    def refresh_list():
        items = scan_external_projects()
        items_select.options = [(f"{slug} — {path}", (slug, path)) for slug, path in items]

    def on_refresh(_):
        refresh_list()
        with out:
            clear_output()
            print(f"Encontrados {len(items_select.options)} projetos externos.")

    def on_import(_):
        sel = items_select.value
        if not sel:
            with out:
                clear_output()
                print("Selecione um ou mais projetos.")
            return
        logs = []
        for slug, path in sel:
            logs.append(import_project(slug, Path(path), mode_rb.value))
        with out:
            clear_output()
            for line in logs:
                print(line)

    refresh_btn.on_click(on_refresh)
    import_btn.on_click(on_import)

    refresh_list()
    display(widgets.VBox([
        widgets.HTML("<b>Sincronizar projetos externos</b>"),
        widgets.HBox([refresh_btn, mode_rb]),
        items_select,
        import_btn,
        out,
    ]))
else:
    print("ipywidgets indisponível — sync UI não renderizado.")


## Global 80% zoom and sticky control bars

Applies a 0.8 zoom to outputs and defines utility CSS classes for sticky top/bottom control bars inside a cell.

In [ ]:
# Inject CSS for zoom and sticky bars
try:
    from IPython.display import HTML, display
    css = """
    <style>
    /* 80% zoom on common output containers */
    div.output, .jp-RenderedHTMLCommon, .jp-OutputArea, .vbox, .hbox { zoom: 0.8; }

    /* Sticky helpers: wrap controls inside a container with these classes */
    .sticky-top { position: sticky; top: 0; z-index: 10; background: var(--jp-layout-color1, #fff); padding: 6px 0; border-bottom: 1px solid rgba(0,0,0,0.05); }
    .sticky-bottom { position: sticky; bottom: 0; z-index: 10; background: var(--jp-layout-color1, #fff); padding: 6px 0; border-top: 1px solid rgba(0,0,0,0.05); }

    /* Compact spacing for controls inside notebooks */
    .compact > * { margin-right: 8px !important; }
    </style>
    """
    display(HTML(css))
except Exception as e:
    print("Could not inject CSS:", e)


## Tabs/Accordion compact dashboard

A tabbed UI grouping core sections: Faixas, Arte, To-do, Letras, Mix, Qualidade, Exportar.
- Compact views load/save data from Projects/<slug>.*
- Includes filterable tables (qgrid if available, fallback to text filters).
- Mix tab shows expanded metrics when audio libs are available.

In [ ]:
# Compact tabs dashboard (Faixas, Arte, To-do, Letras, Mix, Qualidade, Exportar)
from pathlib import Path
import os, json
from typing import List, Dict, Any, Optional

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as e:
    widgets = None
    print("ipywidgets not available:", e)

try:
    import pandas as pd
except Exception:
    pd = None

# Optional libs for audio metrics/plots
try:
    import librosa, numpy as np
except Exception:
    librosa = None
    np = None
try:
    import plotly.graph_objects as go
except Exception:
    go = None
try:
    import soundfile as sf
except Exception:
    sf = None

PROJECTS_ROOT = Path("Projects")

# --- Helpers ---

def _load_json(path: Path, default):
    try:
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Failed to read {path}: {e}")
    return default

def _save_json(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def list_projects():
    return sorted([d.name for d in PROJECTS_ROOT.iterdir() if d.is_dir() and (d/"project.json").exists()])


def read_project(slug: str) -> Dict[str, Any]:
    return _load_json(PROJECTS_ROOT/slug/"project.json", {"name": slug, "tracks": [], "tasks": []})


def write_project(slug: str, data: Dict[str, Any]):
    _save_json(PROJECTS_ROOT/slug/"project.json", data)


def read_artwork(slug: str) -> Dict[str, Any]:
    return _load_json(PROJECTS_ROOT/slug/"artwork.json", {"concept": "", "palette": [], "checklist": []})


def write_artwork(slug: str, data: Dict[str, Any]):
    _save_json(PROJECTS_ROOT/slug/"artwork.json", data)


def read_tasks(slug: str) -> List[Dict[str, Any]]:
    return _load_json(PROJECTS_ROOT/slug/"tasks.json", [])


def write_tasks(slug: str, items: List[Dict[str, Any]]):
    _save_json(PROJECTS_ROOT/slug/"tasks.json", items)


def read_lyrics(slug: str) -> Dict[str, str]:
    return _load_json(PROJECTS_ROOT/slug/"lyrics.json", {})


def write_lyrics(slug: str, m: Dict[str, str]):
    _save_json(PROJECTS_ROOT/slug/"lyrics.json", m)


def list_mixes(slug: str) -> List[Path]:
    mixes_dir = PROJECTS_ROOT/slug/"mixes"
    if not mixes_dir.exists():
        return []
    return sorted([p for p in mixes_dir.iterdir() if p.suffix.lower() in (".wav", ".aiff", ".aif", ".mp3", ".flac")])

# --- Filterable table helper ---

def filter_dataframe(df: Optional['pd.DataFrame'], keyword: str, cols: Optional[List[str]] = None):
    if df is None or pd is None:
        return None
    if not keyword:
        return df
    k = keyword.lower()
    subset = df
    if cols:
        subset = subset[cols]
    mask = subset.astype(str).apply(lambda s: s.str.lower().str.contains(k, na=False))
    mask_any = mask.any(axis=1)
    return df[mask_any]

# --- Audio metrics ---

def crest_factor(samples: np.ndarray) -> Optional[Dict[str, float]]:
    if np is None:
        return None
    try:
        peak = np.max(np.abs(samples)) + 1e-12
        rms = np.sqrt(np.mean(samples**2)) + 1e-12
        ratio = peak / rms
        cfdB = 20*np.log10(ratio)
        return {"ratio": float(ratio), "dB": float(cfdB)}
    except Exception:
        return None


def avg_spectrum(samples: np.ndarray, sr: int, n_fft: int = 4096) -> Optional[Dict[str, Any]]:
    if np is None or librosa is None:
        return None
    try:
        S = np.abs(librosa.stft(samples, n_fft=n_fft))
        spec = np.mean(S, axis=1)
        freqs = np.linspace(0, sr/2, len(spec))
        return {"freqs": freqs.tolist(), "spec": spec.tolist()}
    except Exception:
        return None


def tonal_balance(samples: np.ndarray, sr: int) -> Optional[Dict[str, float]]:
    if np is None or librosa is None:
        return None
    try:
        S, _ = librosa.magphase(librosa.stft(samples, n_fft=4096))
        mag = np.mean(S, axis=1)
        freqs = np.linspace(0, sr/2, len(mag))
        low = mag[(freqs>=20)&(freqs<200)].sum()
        mid = mag[(freqs>=200)&(freqs<2000)].sum()
        high = mag[(freqs>=2000)&(freqs<=sr/2)].sum()
        total = low+mid+high + 1e-12
        return {"low": float(low/total), "mid": float(mid/total), "high": float(high/total)}
    except Exception:
        return None

# --- UI ---
if widgets is not None:
    # Global selectors + message
    proj_dd = widgets.Dropdown(options=[("Selecione um projeto", "__none__")] + [(s, s) for s in list_projects()], description="Projeto:")
    msg_out = widgets.Output()

    # Faixas tab
    tracks_filter = widgets.Text(placeholder="Filtrar faixas...", description="Filtro:")
    tracks_table_out = widgets.Output()

    def refresh_tracks():
        tracks_table_out.clear_output()
        if proj_dd.value in (None, "__none__"):
            return
        proj = read_project(proj_dd.value)
        tracks = proj.get("tracks", [])
        if pd is None:
            with tracks_table_out:
                print("Instale pandas para tabela.")
                for i,t in enumerate(tracks,1):
                    print(i, t)
            return
        df = pd.DataFrame(tracks)
        fdf = filter_dataframe(df, tracks_filter.value, cols=[c for c in df.columns if c not in ("path",)])
        with tracks_table_out:
            clear_output()
            display(fdf)

    tracks_filter.observe(lambda c: refresh_tracks(), names='value')

    # Arte tab
    art_concept = widgets.Textarea(placeholder="Conceito de capa...", description="Conceito:")
    art_palette = widgets.Text(placeholder="#RRGGBB, #...", description="Paleta:")
    art_save_btn = widgets.Button(description="Guardar", button_style="success")
    art_out = widgets.Output()

    def load_artwork():
        if proj_dd.value in (None, "__none__"):
            return
        data = read_artwork(proj_dd.value)
        art_concept.value = data.get("concept", "")
        art_palette.value = ", ".join(data.get("palette", []))

    def on_art_save(_):
        if proj_dd.value in (None, "__none__"):
            return
        palette = [s.strip() for s in art_palette.value.split(',') if s.strip()]
        write_artwork(proj_dd.value, {"concept": art_concept.value, "palette": palette})
        with art_out:
            clear_output(); print("Arte salva.")

    art_save_btn.on_click(on_art_save)

    # To-do tab
    task_filter = widgets.Text(placeholder="Filtrar tarefas...", description="Filtro:")
    task_new = widgets.Text(placeholder="Nova tarefa...", description="Nova:")
    task_add = widgets.Button(description="Adicionar", button_style="primary")
    tasks_out = widgets.Output()

    def refresh_tasks():
        tasks_out.clear_output()
        if proj_dd.value in (None, "__none__"):
            return
        items = read_tasks(proj_dd.value)
        if pd is None:
            with tasks_out:
                print("Instale pandas para tabela.")
                for i,t in enumerate(items,1):
                    print(i, t)
            return
        df = pd.DataFrame(items)
        fdf = filter_dataframe(df, task_filter.value)
        with tasks_out:
            clear_output(); display(fdf)

    def on_task_add(_):
        if proj_dd.value in (None, "__none__") or not task_new.value.strip():
            return
        items = read_tasks(proj_dd.value)
        items.append({"title": task_new.value.strip(), "status": "todo"})
        write_tasks(proj_dd.value, items)
        task_new.value = ""
        refresh_tasks()

    task_add.on_click(on_task_add)
    task_filter.observe(lambda c: refresh_tasks(), names='value')

    # Letras tab
    lyrics_select = widgets.Dropdown(options=[("Selecione faixa", "__none__")], description="Faixa:")
    lyrics_text = widgets.Textarea(placeholder="Letra...", description="Texto:", layout=widgets.Layout(height='180px'))
    lyrics_save = widgets.Button(description="Guardar letra", button_style="success")
    lyrics_out = widgets.Output()

    def load_lyrics_tracks():
        if proj_dd.value in (None, "__none__"):
            lyrics_select.options = [("Selecione faixa", "__none__")]
            return
        proj = read_project(proj_dd.value)
        titles = []
        for i, t in enumerate(proj.get("tracks", []), 1):
            title = t.get("title") or t.get("name") or f"Faixa {i}"
            titles.append((title, title))
        lyrics_select.options = [("Selecione faixa", "__none__")] + titles

    def on_lyrics_change(change):
        if change.get('name')=='value' and change['new'] not in (None, "__none__") and proj_dd.value not in (None, "__none__"):
            m = read_lyrics(proj_dd.value)
            lyrics_text.value = m.get(lyrics_select.value, "")

    def on_lyrics_save(_):
        if proj_dd.value in (None, "__none__") or lyrics_select.value in (None, "__none__"):
            return
        m = read_lyrics(proj_dd.value)
        m[lyrics_select.value] = lyrics_text.value
        write_lyrics(proj_dd.value, m)
        with lyrics_out:
            clear_output(); print("Letra salva.")

    lyrics_select.observe(on_lyrics_change, names='value')
    lyrics_save.on_click(on_lyrics_save)

    # Mix tab with metrics
    mix_select = widgets.Dropdown(options=[("Selecione mix", "__none__")], description="Mix:")
    mix_metrics_out = widgets.Output()
    spectrum_out = widgets.Output()

    def refresh_mixes():
        if proj_dd.value in (None, "__none__"):
            mix_select.options = [("Selecione mix", "__none__")]
            return
        options = [(p.name, str(p)) for p in list_mixes(proj_dd.value)]
        mix_select.options = [("Selecione mix", "__none__")] + options

    def on_mix_change(change):
        if change.get('name')=='value' and change['new'] not in (None, "__none__"):
            path = change['new']
            mix_metrics_out.clear_output(); spectrum_out.clear_output()
            if librosa is None or np is None or sf is None:
                with mix_metrics_out:
                    print("Instale librosa e soundfile para métricas.")
                return
            try:
                samples, sr = librosa.load(path, sr=None, mono=True)
                cf = crest_factor(samples)
                tb = tonal_balance(samples, sr)
                with mix_metrics_out:
                    if cf: print(f"Crest factor: {cf['ratio']:.2f} ({cf['dB']:.2f} dB)")
                    if tb: print(f"Tonal balance: low {tb['low']:.2%}, mid {tb['mid']:.2%}, high {tb['high']:.2%}")
                if go is not None:
                    sp = avg_spectrum(samples, sr)
                    if sp:
                        fig = go.Figure()
                        fig.add_trace(go.Scatter(x=sp['freqs'], y=sp['spec'], mode='lines', name='Spectrum'))
                        fig.update_layout(xaxis_type='log', xaxis_title='Hz (log)', yaxis_title='Magnitude')
                        with spectrum_out:
                            clear_output(); display(fig)
            except Exception as e:
                with mix_metrics_out:
                    print("Erro ao ler mix:", e)

    mix_select.observe(on_mix_change, names='value')

    # Qualidade tab (simple score example)
    quality_out = widgets.Output()
    def refresh_quality():
        quality_out.clear_output()
        if proj_dd.value in (None, "__none__"):
            return
        proj = read_project(proj_dd.value)
        score = 0
        if proj.get('tracks'): score += 30
        if read_tasks(proj_dd.value): score += 20
        if list_mixes(proj_dd.value): score += 25
        if read_artwork(proj_dd.value).get('concept'): score += 25
        with quality_out:
            print(f"Qualidade (heurística): {score}/100")

    # Export tab (PDF via earlier utilities if present)
    export_out = widgets.Output()
    export_pdf_btn = widgets.Button(description="Exportar PDF", button_style="primary")

    def on_export_pdf(_):
        export_out.clear_output()
        try:
            from __main__ import project_markdown, export_pdf_from_markdown
        except Exception:
            with export_out:
                print("Funções de export (project_markdown/export_pdf_from_markdown) não disponíveis nesta sessão.")
            return
        if proj_dd.value in (None, "__none__"):
            with export_out:
                print("Selecione um projeto.")
            return
        md = project_markdown(proj_dd.value)
        out_pdf = PROJECTS_ROOT/proj_dd.value/"exports"/f"{proj_dd.value}-report.pdf"
        res = export_pdf_from_markdown(md, out_pdf)
        with export_out:
            if res.exists(): print("PDF gerado:", res)
            else: print("Não foi possível gerar PDF.")

    export_pdf_btn.on_click(on_export_pdf)

    # Tabs wiring
    def on_project_change(change):
        if change.get('name')=='value':
            refresh_tracks(); load_artwork(); refresh_tasks(); load_lyrics_tracks(); refresh_mixes(); refresh_quality()

    proj_dd.observe(on_project_change, names='value')

    tabs = widgets.Tab(children=[
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Controlo rápido — Faixas</div>'),
            tracks_filter,
            tracks_table_out,
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Controlo rápido — Arte</div>'),
            art_concept, art_palette, art_save_btn, art_out
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Controlo rápido — To-do</div>'),
            widgets.HBox([task_filter, task_new, task_add]),
            tasks_out
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Controlo rápido — Letras</div>'),
            lyrics_select, lyrics_text, lyrics_save, lyrics_out
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Controlo rápido — Mix</div>'),
            mix_select, mix_metrics_out, spectrum_out
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Sinal de qualidade</div>'),
            quality_out
        ]),
        widgets.VBox([
            widgets.HTML('<div class="sticky-top compact">Exportar</div>'),
            export_pdf_btn, export_out
        ]),
    ])
    for i, title in enumerate(["Faixas", "Arte", "To-do", "Letras", "Mix", "Qualidade", "Exportar"]):
        tabs.set_title(i, title)

    display(widgets.VBox([
        widgets.HTML('<div class="sticky-top compact">Dashboard compacto</div>'),
        proj_dd,
        tabs,
        widgets.HTML('<div class="sticky-bottom">Controles fixos</div>'),
        msg_out
    ]))
else:
    print("ipywidgets indisponível — Tabs UI não renderizado.")


## Coletânea (compilação de singles)

Selecione projetos existentes (singles) e combine as faixas num novo projeto (Mixtape/LP). Pode copiar os arquivos de áudio ou manter apenas referências.

In [ ]:
# Compilation tool: combine singles into Mixtape/LP
from pathlib import Path
import os, json, shutil
from typing import List, Dict, Any

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    widgets = None

PROJECTS_ROOT = Path("Projects")


def list_projects():
    return sorted([d.name for d in PROJECTS_ROOT.iterdir() if d.is_dir() and (d/"project.json").exists()])


def read_proj(slug: str) -> Dict[str, Any]:
    p = PROJECTS_ROOT/slug/"project.json"
    if p.exists():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {"name": slug, "tracks": []}


def write_proj(slug: str, data: Dict[str, Any]):
    out = PROJECTS_ROOT/slug/"project.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def combine_singles(slugs: List[str], new_slug: str, new_name: str, kind: str, copy_audio: bool) -> str:
    if not new_slug:
        raise ValueError("Forneça um slug para o novo projeto.")
    dst_dir = PROJECTS_ROOT/new_slug
    if dst_dir.exists():
        raise FileExistsError(f"Projeto destino já existe: {dst_dir}")
    dst_dir.mkdir(parents=True, exist_ok=True)

    combined: Dict[str, Any] = {"name": new_name or new_slug, "type": kind, "tracks": []}
    dst_tracks = dst_dir/"tracks"
    if copy_audio:
        dst_tracks.mkdir(exist_ok=True)

    for slug in slugs:
        proj = read_proj(slug)
        for t in proj.get("tracks", []):
            nt = dict(t)
            src_path = None
            # Track path may live under source project's tracks dir
            if t.get("path"):
                src_path = Path(t["path"]) if os.path.isabs(t["path"]) else (PROJECTS_ROOT/slug/"tracks"/t["path"]).resolve()
                if copy_audio and src_path.exists():
                    dst_file = dst_tracks/src_path.name
                    try:
                        shutil.copy2(src_path, dst_file)
                        nt["path"] = str(dst_file)
                    except Exception:
                        nt["path"] = str(src_path)
                else:
                    nt["path"] = str(src_path)
            combined["tracks"].append(nt)

    write_proj(new_slug, combined)
    return f"Criado {new_slug} com {len(combined['tracks'])} faixas."


if widgets is not None:
    source_select = widgets.SelectMultiple(options=[(s, s) for s in list_projects()], description="Singles:", rows=10)
    new_slug = widgets.Text(placeholder="novo-slug", description="Slug:")
    new_name = widgets.Text(placeholder="Nome do projeto", description="Nome:")
    kind_dd = widgets.Dropdown(options=[("Mixtape", "Mixtape"), ("LP", "LP"), ("EP", "EP")], value="Mixtape", description="Tipo:")
    copy_chk = widgets.Checkbox(value=False, description="Copiar arquivos de áudio")
    run_btn = widgets.Button(description="Criar coletânea", button_style="success")
    out = widgets.Output()

    def on_run(_):
        out.clear_output()
        slugs = list(source_select.value)
        if not slugs:
            with out: print("Selecione pelo menos um projeto."); return
        try:
            msg = combine_singles(slugs, new_slug.value.strip(), new_name.value.strip(), kind_dd.value, copy_chk.value)
            with out: print(msg)
        except Exception as e:
            with out: print("Erro:", e)

    run_btn.on_click(on_run)
    display(widgets.VBox([
        widgets.HTML('<b>Coletânea de singles</b>'),
        source_select,
        widgets.HBox([new_slug, new_name]),
        widgets.HBox([kind_dd, copy_chk]),
        run_btn,
        out,
    ]))
else:
    print("ipywidgets indisponível — ferramenta de coletânea não renderizada.")
